In [1]:
from pathlib import Path
import numpy as np
if not hasattr(np, "int"): np.int = int      
import pretty_midi as pm
import os, glob, warnings, collections
import regex as re

/opt/anaconda3/envs/ml_hw/lib/python3.11/site-packages/pretty_midi/instrument.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
def get_files(folder_path):
    item_list = []
    for item in folder_path.iterdir():
        item_list.append(item)
    return item_list

In [106]:
folder_path = Path("clean_midi")
INTERESTED_BANDS = ["The Doors"] #,"AC DC", "Pink Floyd", "Queen"]

def get_midi_list(
        folder_path = Path("clean_midi"),
        BAND_LIST = None 
        ):
    midi_path_list = []
    for band in get_files(folder_path):
        if(band.name in BAND_LIST):
            band_path = Path(f"{folder_path}/{band.name}")
            for song in get_files(band_path):
                midi_path_list.append(f"{band_path}/{song.name}")
                #print(song.name)
    
    return midi_path_list

In [107]:
def get_midi_notes(midi_list):
    for midi_path in midi_list:
        print(midi_path)
        midi = pm.PrettyMIDI(midi_path)
        for instrument in midi.instruments:
            print(instrument.name)
            for note in instrument.notes:
                print(note)
        break

In [108]:
def dedup_midis(midi_list):
    deduped_midis = []
    pattern = r"\.\d*\.mid"
    for midi_path in midi_list:
        if re.search(pattern, midi_path):
            continue
        else:
            deduped_midis.append(midi_path)

    return deduped_midis

In [109]:
## deduplicating the tracks, as we have a bunch of numbereed tracks per song

midi_list = get_midi_list(folder_path, INTERESTED_BANDS)
print(f"tracks before dedup: {len(midi_list)}")
midi_list = dedup_midis(midi_list)
print(f"tracks after dedup: {len(midi_list)}")

tracks before dedup: 49
tracks after dedup: 17


In [110]:
from collections import Counter
def get_program_count(midi_list):
    all_programs = []
    for midi_path in midi_list:
        midi = pm.PrettyMIDI(midi_path)
        notes = []
        for inst in midi.instruments:
            all_programs.append(inst.program)
    
    return Counter(all_programs)

print(get_program_count(midi_list))

Counter({np.int64(0): 41, np.int64(18): 10, np.int64(16): 7, np.int64(33): 7, np.int64(27): 7, np.int64(26): 5, np.int64(29): 4, np.int64(122): 4, np.int64(30): 3, np.int64(35): 3, np.int64(6): 3, np.int64(19): 2, np.int64(65): 2, np.int64(24): 2, np.int64(88): 2, np.int64(4): 2, np.int64(61): 2, np.int64(25): 2, np.int64(17): 2, np.int64(32): 1, np.int64(5): 1, np.int64(22): 1, np.int64(20): 1, np.int64(80): 1, np.int64(2): 1, np.int64(37): 1, np.int64(49): 1, np.int64(94): 1, np.int64(34): 1, np.int64(89): 1, np.int64(7): 1, np.int64(119): 1, np.int64(1): 1, np.int64(3): 1, np.int64(125): 1, np.int64(21): 1})


In [ ]:
TARGET_PROGRAMS = {18, 16, 33}   # rock organ(18) is present in 10 tracks, enough data to learn patterns
# {29, 30}                    # Overdriven + Distortion Guitar
GRID_PER_BEAT   = 4                            # 16th-note grid
MAX_OFFSET = 16                                # cap silence between notes (1 bar)
MAX_DUR    = 16


## extracting the token sequences
def extract_token_stream(midi_path):
    midi = pm.PrettyMIDI(midi_path)
    notes = []
    for inst in midi.instruments:
        ## not using drums as they have a different index, we restrict the token sequences to >=100 per instrunent
        ## also restricted to a few instruments for a cleaner sound.
        if inst.is_drum or len(inst.notes) <= 100 or inst.program not in TARGET_PROGRAMS:
            continue
        notes.extend(inst.notes)
    if not notes:
        return None
    notes.sort(key=lambda n: (n.start, n.pitch))

    # seconds -> 16th-note grid using the tempo map
    beats   = midi.get_beats()          # array of beat times in seconds
    sec_per_beat = np.median(np.diff(beats)) if len(beats) > 1 else 0.5
    step    = sec_per_beat / GRID_PER_BEAT

    tokens, prev_start_step = [], 0
    for n in notes:
        s = int(round(n.start / step))
        d = max(1, min(MAX_DUR, int(round((n.end - n.start) / step))))
        o = max(0, min(MAX_OFFSET, s - prev_start_step))
        tokens.append((n.pitch, d, o))
        prev_start_step = s
    return tokens #, program_list

In [122]:
all_streams = []
## putting pretty midi reads in a try catch, as some midis might be corrupted
for p in midi_list:
    try:
        s = extract_token_stream(p)
        if s: all_streams.append(s)
    except Exception as e:
        pass
    #break

print(len(all_streams))

13


In [123]:
## addding pads, beginning of sequence and end of sequence tokens

counter = collections.Counter(tok for s in all_streams for tok in s)
itos = ["<PAD>", "<BOS>", "<EOS>"] + [t for t, _ in counter.most_common()]
stoi = {t: i for i, t in enumerate(itos)}
print(f"{len(all_streams)} songs, vocab size {len(itos)}")

# encode -> integer sequences for the LSTM
encoded = [[stoi["<BOS>"]] + [stoi[t] for t in s] + [stoi["<EOS>"]] for s in all_streams]

# and slice into fixed-length windows for training, we train on 128 sequences
SEQ = 128
xs, ys = [], []
for e in encoded:
    for i in range(len(e) - SEQ - 1):
        xs.append(e[i:i+SEQ]); ys.append(e[i+1:i+SEQ+1])
X = np.array(xs); Y = np.array(ys)
print(X.shape, Y.shape)

13 songs, vocab size 1108
(11759, 128) (11759, 128)


In [124]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")
print("device:", device)

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
EOS_ID = stoi["<EOS>"]
VOCAB  = len(itos)

## each window ends in <EOS> at most once
## and so that short songs don't dominate via overlap. we run a stride to cut redundancy.
STRIDE = 16
xs, ys = [], []
for e in encoded:
    if len(e) < SEQ + 1:
        # pad the song out to one full window so it isn't dropped entirely
        e = e + [PAD_ID] * (SEQ + 1 - len(e))
    for i in range(0, len(e) - SEQ - 1, STRIDE):
        xs.append(e[i:i+SEQ])
        ys.append(e[i+1:i+SEQ+1])

X = torch.tensor(xs, dtype=torch.long)
Y = torch.tensor(ys, dtype=torch.long)
print("training tensor:", X.shape, Y.shape)

class MidiTokDataset(Dataset):
    def __init__(self, X, Y): self.X, self.Y = X, Y
    def __len__(self):        return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]

ds = MidiTokDataset(X, Y)
n_val = max(1, int(0.05 * len(ds)))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val],
                                generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, drop_last=False)
print(f"train batches: {len(train_loader)}   val batches: {len(val_loader)}")


device: mps
training tensor: torch.Size([741, 128]) torch.Size([741, 128])
train batches: 11   val batches: 1


In [125]:
class MidiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden=256, n_layers=2, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.lstm  = nn.LSTM(embed_dim, hidden, num_layers=n_layers,
                             dropout=dropout if n_layers > 1 else 0.0,
                             batch_first=True)
        self.drop  = nn.Dropout(dropout)
        self.head  = nn.Linear(hidden, vocab_size)

    def forward(self, x, hidden=None):
        # x: (B, T) long
        emb = self.embed(x)                     # (B, T, E)
        out, hidden = self.lstm(emb, hidden)    # (B, T, H)
        logits = self.head(self.drop(out))      # (B, T, V)
        return logits, hidden

model = MidiLSTM(VOCAB).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"params: {n_params/1e6:.2f}M")


MidiLSTM(
  (embed): Embedding(1108, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.2)
  (drop): Dropout(p=0.2, inplace=False)
  (head): Linear(in_features=256, out_features=1108, bias=True)
)
params: 1.35M


In [126]:
EPOCHS = 10
LR     = 3e-3
CLIP   = 1.0

optim  = torch.optim.Adam(model.parameters(), lr=LR)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def run_epoch(loader, train=True):
    model.train(train)
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits, _ = model(xb)
        loss = loss_fn(logits.reshape(-1, VOCAB), yb.reshape(-1))
        if train:
            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            optim.step()
        total += loss.item() * xb.size(0)
        n     += xb.size(0)
    return total / n

history = []
for ep in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va = run_epoch(val_loader, train=False)
    sched.step()
    history.append((tr, va))
    print(f"epoch {ep:2d}  train {tr:.3f}  val {va:.3f}  "
          f"train_ppl {np.exp(tr):7.1f}  val_ppl {np.exp(va):7.1f}")

epoch  1  train 5.700  val 4.693  train_ppl   298.9  val_ppl   109.2
epoch  2  train 4.055  val 3.827  train_ppl    57.7  val_ppl    45.9
epoch  3  train 2.979  val 3.073  train_ppl    19.7  val_ppl    21.6
epoch  4  train 2.221  val 2.596  train_ppl     9.2  val_ppl    13.4
epoch  5  train 1.710  val 2.224  train_ppl     5.5  val_ppl     9.2
epoch  6  train 1.395  val 2.038  train_ppl     4.0  val_ppl     7.7
epoch  7  train 1.199  val 1.879  train_ppl     3.3  val_ppl     6.5
epoch  8  train 1.086  val 1.821  train_ppl     3.0  val_ppl     6.2
epoch  9  train 1.024  val 1.778  train_ppl     2.8  val_ppl     5.9
epoch 10  train 1.001  val 1.771  train_ppl     2.7  val_ppl     5.9


In [127]:
@torch.no_grad()
def generate(model, max_tokens=400, temperature=1.0, top_k=40, seed_tokens=None):
    model.eval()
    if seed_tokens is None:
        seed_tokens = [BOS_ID]
    x = torch.tensor([seed_tokens], dtype=torch.long, device=device)
    hidden = None
    out_ids = list(seed_tokens)

    # warm up the hidden state on the seed
    _, hidden = model(x, hidden)
    cur = x[:, -1:]                              # last token

    for _ in range(max_tokens):
        logits, hidden = model(cur, hidden)
        logits = logits[:, -1, :] / max(1e-6, temperature)

        # mask special tokens we never want to emit mid-stream
        logits[0, PAD_ID] = -float("inf")
        logits[0, BOS_ID] = -float("inf")

        if top_k is not None and top_k > 0:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        nid = nxt.item()
        out_ids.append(nid)
        if nid == EOS_ID:
            break
        cur = nxt
    return out_ids

sample_ids = generate(model, max_tokens=500, temperature=1.0, top_k=40)
print(f"generated {len(sample_ids)} tokens, first 12 ids: {sample_ids[:12]}")
# decode ids -> human-readable symbols (special tokens are strings, notes are tuples)
sample_syms = [itos[i] for i in sample_ids]
print("first 8 symbols:", sample_syms[:8])


generated 501 tokens, first 12 ids: [1, 89, 25, 21, 264, 170, 600, 273, 14, 96, 96, 189]
first 8 symbols: ['<BOS>', (86, 2, 0), (76, 4, 0), (80, 2, 0), (89, 1, 0), (81, 1, 2), (62, 4, 1), (84, 2, 2)]


In [128]:
import pretty_midi as pmlib

def tokens_to_midi(ids,
                   out_path="generated.mid",
                   bpm=120,
                   program=18,           # 18 = Rock Organ; pick to match your band
                   default_velocity=90):
    """Turn a list of integer ids back into a playable .mid file.

    Each non-special id maps to a (pitch, dur_16ths, offset_16ths_from_prev_start)
    triple via itos. We rebuild a single-instrument MIDI by walking the stream,
    advancing a time cursor by `offset` 16th-notes per token, and placing a note
    from cursor to cursor + duration.
    """
    midi = pmlib.PrettyMIDI(initial_tempo=bpm)
    inst = pmlib.Instrument(program=program, name="generated")
    midi.instruments.append(inst)

    sec_per_beat = 60.0 / bpm
    step         = sec_per_beat / GRID_PER_BEAT      # one 16th-note in seconds
    cursor_steps = 0

    for tid in ids:
        sym = itos[tid]
        if not isinstance(sym, tuple):
            # special token: skip BOS/PAD; stop on EOS
            if sym == "<EOS>":
                break
            continue
        pitch, dur, offset = sym
        cursor_steps += offset
        start_s = cursor_steps * step
        end_s   = (cursor_steps + max(1, dur)) * step
        if 0 <= pitch < 128 and end_s > start_s:
            inst.notes.append(pmlib.Note(
                velocity=default_velocity,
                pitch=int(pitch),
                start=float(start_s),
                end=float(end_s),
            ))

    midi.write(out_path)
    print(f"wrote {len(inst.notes)} notes -> {out_path}  ({inst.notes[-1].end:.1f}s)")
    return midi

generated_midi = tokens_to_midi(sample_ids, out_path="generated_doors.mid",
                                bpm=120, program=18)


wrote 500 notes -> generated_doors.mid  (67.8s)
